# PhonePe Pulse — Transaction Insights & Business Analysis

**Project Type:** Exploratory Data Analysis (EDA)  
**Domain:** Finance / Digital Payments  
**Tools Used:** Python, PostgreSQL, Plotly, Pandas, Streamlit  
**Data Source:** PhonePe Pulse GitHub Repository  

---

## Project Objective

PhonePe is one of India's leading digital payment platforms processing billions of 
transactions across 36 states and union territories. This analysis explores:

- Transaction patterns across states, districts, and payment categories
- Device brand preferences and user engagement behavior
- Insurance adoption and penetration across India
- Geographic hotspots at district and pin code level
- Year over year growth trends from 2018 to 2024

The goal is to extract actionable business insights that can help PhonePe make 
data driven decisions on market expansion, user retention, and product development.

---

## Data Quality Issues Identified

The following issues were found during data exploration and are documented here
for transparency:

1. **Insurance data** starts from 2020 Q2, not 2018. Early quarters are missing.
2. **Hyderabad district** shows an anomalous spike in 2022 followed by a drop in 
   2023 — likely due to district boundary reclassification.
3. **Ahmedabad district** appears with two spellings in the data — "ahmedabad" and 
   "ahmadabad" — representing the same geographic area.
4. **Device brand data** drops sharply after 2021 — likely a change in PhonePe's 
   data reporting methodology.
5. **Aggregated transaction table** contains an 'india' row that is a national 
   summary — excluded from all state level calculations to prevent double counting.

In [42]:
# ============================================================
# IMPORTS
# Purpose: Load all libraries needed for the entire notebook.
# We import everything upfront so there are no missing module
# errors midway through the analysis.
# ============================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# DATABASE CONNECTION
# Purpose: Create one engine reused across all queries.
# We use SQLAlchemy which gives us a clean interface to
# run SQL queries and get results directly as DataFrames.
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "1234"  
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "phonepe_pulse"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print(" Connected to:", result.fetchone()[0])

 Connected to: PostgreSQL 18.2 on x86_64-windows, compiled by msvc-19.44.35222, 64-bit


---
## 1. Data Overview

Before analysis, we inspect all 9 tables to understand their structure,
size, and content. This is critical to ensure we know exactly what data
we're working with before drawing any conclusions.

We have 3 data categories × 3 geographic levels = 9 tables:

| Category | Aggregated (State) | Map (District) | Top (Pincode) |
|---|---|---|---|
| Transaction | aggregated_transaction | map_transaction | top_transaction |
| User | aggregated_user | map_user | top_user |
| Insurance | aggregated_insurance | map_insurance | top_insurance |

In [43]:
# ============================================================
# TABLE OVERVIEW
# Purpose: Check row counts and column names for all 9 tables
# so we have a complete picture of the dataset before analysis.
# ============================================================

tables = [
    'aggregated_transaction', 'aggregated_user', 'aggregated_insurance',
    'map_transaction', 'map_user', 'map_insurance',
    'top_transaction', 'top_user', 'top_insurance'
]

print("=" * 60)
print(f"{'Table':<30} {'Rows':>10} {'Columns':>10}")
print("=" * 60)

for table in tables:
    with engine.connect() as conn:
        row_count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).fetchone()[0]
        cols = conn.execute(text(f"""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_name = '{table}'
        """)).fetchall()
        col_names = [c[0] for c in cols]
        print(f"{table:<30} {row_count:>10,} {len(col_names):>10}")
        print(f"  Columns: {', '.join(col_names)}")
        print()

Table                                Rows    Columns
aggregated_transaction              5,174          6
  Columns: year, quarter, transaction_count, transaction_amount, state, transaction_type

aggregated_user                     6,919          8
  Columns: device_percentage, year, quarter, registered_users, app_opens, device_count, state, device_brand

aggregated_insurance                  701          6
  Columns: year, quarter, transaction_count, transaction_amount, state, transaction_type

map_transaction                    20,604          6
  Columns: year, quarter, transaction_count, transaction_amount, state, district

map_user                           20,608          6
  Columns: year, quarter, registered_users, app_opens, state, district

map_insurance                      13,876          6
  Columns: year, quarter, transaction_count, transaction_amount, state, district

top_transaction                    18,295          7
  Columns: quarter, transaction_count, transaction_

### Sample Data Preview
We look at the first few rows of key tables to understand
the actual values and formats stored in each column.

In [44]:
# ============================================================
# SAMPLE DATA PREVIEW
# Purpose: Look at actual rows from key tables to understand
# data formats, value ranges, and spot any obvious issues.
# ============================================================

def preview_table(table_name, rows=3):
    """
    Fetches a small sample from a table and displays it.
    Using a function avoids repeating the same query code
    for every table -- addresses 'messy code' feedback.
    """
    with engine.connect() as conn:
        df = pd.read_sql(
            text(f"SELECT * FROM {table_name} LIMIT {rows}"),
            conn
        )
    print(f"\n{'='*60}")
    print(f"TABLE: {table_name}")
    print(f"{'='*60}")
    print(df.to_string(index=False))
    return df

# Preview the three most important tables
preview_table('aggregated_transaction')
preview_table('aggregated_user')
preview_table('map_transaction')


TABLE: aggregated_transaction
state  year  quarter         transaction_type  transaction_count  transaction_amount
india  2018        1 Recharge & bill payments           72550406        1.447271e+10
india  2018        1    Peer-to-peer payments           46982705        1.472459e+11
india  2018        1        Merchant payments            5368669        4.656679e+09

TABLE: aggregated_user
state  year  quarter  registered_users  app_opens device_brand  device_count  device_percentage
india  2018        1          46877867          0       Xiaomi      11926334           0.254413
india  2018        1          46877867          0      Samsung       9609401           0.204988
india  2018        1          46877867          0         Vivo       5894293           0.125737

TABLE: map_transaction
                    state  year  quarter                          district  transaction_count  transaction_amount
andaman-&-nicobar-islands  2018        1 north and middle andaman district         

,state,year,quarter,district,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,north and middle andaman district,442,9.316631e+05
1,andaman-&-nicobar-islands,2018,1,south andaman district,5688,1.256025e+07
2,andaman-&-nicobar-islands,2018,1,nicobars district,528,1.139849e+06


In [45]:
# ============================================================
# DATA RANGE OVERVIEW
# Purpose: Confirm the time period and geographic coverage
# of the dataset so we know the boundaries of our analysis.
# ============================================================

def get_data_range(table_name, extra_col=None):
    """
    Returns year range, quarter range and unique state count
    for any table. Optional extra_col shows unique values
    for that column (e.g. transaction_type, device_brand).
    """
    with engine.connect() as conn:
        result = conn.execute(text(f"""
            SELECT 
                MIN(year) AS start_year,
                MAX(year) AS end_year,
                MIN(quarter) AS min_quarter,
                MAX(quarter) AS max_quarter,
                COUNT(DISTINCT state) AS unique_states
            FROM {table_name}
        """)).fetchone()
        
        print(f"\n{table_name}")
        print(f"  Years       : {result[0]} to {result[1]}")
        print(f"  Quarters    : Q{result[2]} to Q{result[3]}")
        print(f"  States      : {result[4]}")
        
        if extra_col:
            extra = conn.execute(text(f"""
                SELECT DISTINCT {extra_col} 
                FROM {table_name} 
                ORDER BY {extra_col}
            """)).fetchall()
            values = [r[0] for r in extra]
            print(f"  {extra_col}: {', '.join(str(v) for v in values)}")

get_data_range('aggregated_transaction', 'transaction_type')
get_data_range('aggregated_user', 'device_brand')
get_data_range('aggregated_insurance')
get_data_range('map_transaction')
get_data_range('map_user')


aggregated_transaction
  Years       : 2018 to 2024
  Quarters    : Q1 to Q4
  States      : 37
  transaction_type: Financial Services, Merchant payments, Others, Peer-to-peer payments, Recharge & bill payments

aggregated_user
  Years       : 2018 to 2022
  Quarters    : Q1 to Q4
  States      : 37
  device_brand: Apple, Asus, COOLPAD, Gionee, HMD Global, Huawei, Infinix, Lava, Lenovo, Lyf, Micromax, Motorola, OnePlus, Oppo, Others, Realme, Samsung, Tecno, Vivo, Xiaomi

aggregated_insurance
  Years       : 2020 to 2024
  Quarters    : Q1 to Q4
  States      : 37

map_transaction
  Years       : 2018 to 2024
  Quarters    : Q1 to Q4
  States      : 36

map_user
  Years       : 2018 to 2024
  Quarters    : Q1 to Q4
  States      : 36


---
# 2. Case Study 1 — Decoding Transaction Dynamics

## Business Context
PhonePe processes billions of transactions across 5 payment categories.
Understanding which categories dominate, which states lead, and how growth
has trended over time is fundamental to PhonePe's business strategy.

We analyze:
- Payment category distribution nationally
- Top performing states by volume and value
- Year over year growth trajectory
- Quarterly seasonality patterns

In [46]:
# ============================================================
# CASE STUDY 1 - DATA LOADING
# Purpose: Load all data needed for Case Study 1 charts
# upfront in one place. This keeps chart code clean --
# each chart function just uses these DataFrames directly
# without repeating SQL queries.
# ============================================================

with engine.connect() as conn:

    # Query 1 data -- payment category distribution
    df_cat = pd.read_sql(text("""
        SELECT 
            transaction_type,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount,
            ROUND(
                100.0 * SUM(transaction_count) / SUM(SUM(transaction_count)) OVER(),
                2
            ) AS percentage_share
        FROM aggregated_transaction
        WHERE state != 'india'
        GROUP BY transaction_type
        ORDER BY total_transactions DESC
    """), conn)

    # Query 2 data -- top states
    df_states = pd.read_sql(text("""
        SELECT
            state,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount,
            ROUND(AVG(transaction_amount)::NUMERIC, 2) AS avg_transaction_value
        FROM aggregated_transaction
        WHERE state != 'india'
        GROUP BY state
        ORDER BY total_transactions DESC
        LIMIT 10
    """), conn)

    # Query 3 data -- year over year growth
    df_yoy = pd.read_sql(text("""
        SELECT
            year,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount,
            ROUND(
                100.0 * (SUM(transaction_count) - LAG(SUM(transaction_count)) OVER (ORDER BY year))
                / NULLIF(LAG(SUM(transaction_count)) OVER (ORDER BY year), 0),
                2
            ) AS yoy_growth_percent
        FROM aggregated_transaction
        WHERE state != 'india'
        GROUP BY year
        ORDER BY year
    """), conn)

    # Extra query -- quarterly trends for seasonality analysis
    df_quarterly = pd.read_sql(text("""
        SELECT
            year,
            quarter,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount
        FROM aggregated_transaction
        WHERE state != 'india'
        GROUP BY year, quarter
        ORDER BY year, quarter
    """), conn)

print(" Case Study 1 data loaded successfully")
print(f"  Category data    : {len(df_cat)} rows")
print(f"  State data       : {len(df_states)} rows")
print(f"  YoY growth data  : {len(df_yoy)} rows")
print(f"  Quarterly data   : {len(df_quarterly)} rows")

 Case Study 1 data loaded successfully
  Category data    : 5 rows
  State data       : 10 rows
  YoY growth data  : 7 rows
  Quarterly data   : 28 rows


### Chart 1 — Payment Category Distribution
**Chart type:** Pie chart + Bar chart side by side  
**Why this chart:** We need to show both the proportional share (pie) and 
the absolute difference in values (bar) simultaneously. A pie chart alone 
loses the magnitude difference. A bar chart alone loses the part-to-whole 
relationship. Together they tell the complete story.

In [47]:
# ============================================================
# CHART 1 - Payment Category Distribution
# Shows: Which payment categories dominate PhonePe by
#        transaction count and their percentage share
# Insight target: Merchant payments lead in volume but
#                 P2P leads in value
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Transaction Count Share by Category',
        'Total Transaction Amount by Category (₹ Trillion)'
    ),
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

# --- Left: Pie chart for transaction count share ---
fig.add_trace(
    go.Pie(
        labels=df_cat['transaction_type'],
        values=df_cat['total_transactions'],
        hole=0.4,  # donut style -- easier to read than full pie
        textinfo='label+percent',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Bar chart for transaction amount ---
# Convert to trillions for readable axis labels
df_cat['amount_trillion'] = df_cat['total_amount'] / 1e12

fig.add_trace(
    go.Bar(
        x=df_cat['transaction_type'],
        y=df_cat['amount_trillion'],
        marker_color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A'],
        text=df_cat['amount_trillion'].round(1),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'PhonePe Payment Category Analysis — Volume vs Value',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark'
)

fig.update_xaxes(tickangle=15, row=1, col=2)
fig.update_yaxes(title_text='Amount (₹ Trillion)', row=1, col=2)

fig.show()

**Observation:** Merchant payments dominate transaction count at 55.35% 
but Peer-to-peer payments dominate transaction value at ₹266 trillion — 
4x more than merchant payments despite having fewer transactions.

**Business Insight:** PhonePe users make frequent small merchant payments 
(groceries, utilities) but move large amounts through P2P transfers. 
This split personality of the platform means two distinct strategies are 
needed — volume growth for merchant payments, value growth for P2P.

**Business Impact:** If PhonePe introduces a fee on high value P2P 
transactions above a threshold, the revenue potential is enormous given 
₹266 trillion flows through this category. Merchant payments growth 
should focus on onboarding more small merchants in tier 2 and tier 3 cities.

### Chart 2 — Top 10 States by Transaction Volume and Value
**Chart type:** Horizontal bar chart with dual comparison  
**Why this chart:** We need to compare two metrics (volume and value) 
across 10 states simultaneously. Horizontal bars are easier to read 
when state names are long. Showing both metrics side by side reveals 
the volume vs value gap between states.

In [48]:
# ============================================================
# CHART 2 - Top 10 States by Transaction Volume and Value
# Shows: Which states lead in transaction count vs amount
# Insight target: Maharashtra leads volume but Telangana
#                 leads value -- different market behaviors
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Top 10 States — Transaction Count (Billions)',
        'Top 10 States — Transaction Amount (₹ Trillion)'
    )
)

# Convert to readable units
df_states['transactions_billion'] = df_states['total_transactions'] / 1e9
df_states['amount_trillion'] = df_states['total_amount'] / 1e12

# Clean state names for display
df_states['state_clean'] = df_states['state'].str.replace('-', ' ').str.title()

# Sort separately for each chart
df_vol = df_states.sort_values('transactions_billion', ascending=True)
df_val = df_states.sort_values('amount_trillion', ascending=True)

# --- Left: Transaction count ---
fig.add_trace(
    go.Bar(
        y=df_vol['state_clean'],
        x=df_vol['transactions_billion'],
        orientation='h',
        marker_color='#636EFA',
        text=df_vol['transactions_billion'].round(1),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Transaction amount ---
fig.add_trace(
    go.Bar(
        y=df_val['state_clean'],
        x=df_val['amount_trillion'],
        orientation='h',
        marker_color='#EF553B',
        text=df_val['amount_trillion'].round(1),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'Top 10 States — Transaction Volume vs Value Comparison',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark'
)

fig.update_xaxes(title_text='Transactions (Billions)', row=1, col=1)
fig.update_xaxes(title_text='Amount (₹ Trillion)', row=1, col=2)

fig.show()

### Chart 3 — Year Over Year Transaction Growth
**Chart type:** Dual axis line + bar chart  
**Why this chart:** We need to show two different things simultaneously —
absolute transaction volume (bar) and growth rate percentage (line).
A single axis chart would make one metric unreadable because their 
scales are completely different. Dual axis solves this cleanly.

In [49]:
# ============================================================
# CHART 3 - Year Over Year Transaction Growth
# Shows: Absolute transaction volume growth AND growth rate
#        percentage on the same chart
# Insight target: Growth rate is declining but absolute
#                 numbers are exploding -- market maturity
# ============================================================

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Convert to billions for readable axis
df_yoy['transactions_billion'] = df_yoy['total_transactions'] / 1e9

# --- Bars: Absolute transaction volume ---
fig.add_trace(
    go.Bar(
        x=df_yoy['year'],
        y=df_yoy['transactions_billion'],
        name='Total Transactions (Billions)',
        marker_color='#636EFA',
        text=df_yoy['transactions_billion'].round(1),
        textposition='outside'
    ),
    secondary_y=False
)

# --- Line: YoY growth rate ---
# Drop 2018 row since it has no growth rate (NULL)
df_yoy_growth = df_yoy.dropna(subset=['yoy_growth_percent'])

fig.add_trace(
    go.Scatter(
        x=df_yoy_growth['year'],
        y=df_yoy_growth['yoy_growth_percent'],
        name='YoY Growth Rate (%)',
        mode='lines+markers+text',
        line=dict(color='#EF553B', width=3),
        marker=dict(size=10),
        text=df_yoy_growth['yoy_growth_percent'].astype(str) + '%',
        textposition='top center'
    ),
    secondary_y=True
)

fig.update_layout(
    title={
        'text': 'PhonePe Transaction Growth 2018—2024',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

fig.update_xaxes(title_text='Year')
fig.update_yaxes(title_text='Total Transactions (Billions)', secondary_y=False)
fig.update_yaxes(title_text='YoY Growth Rate (%)', secondary_y=True)

fig.show()

### Chart 4 — Quarterly Seasonality Patterns
**Chart type:** Line chart with multiple lines (one per year)  
**Why this chart:** We want to see if transaction volume follows a 
seasonal pattern within each year. Multiple lines — one per year — 
lets us compare the same quarter across different years simultaneously. 
This reveals both growth over time and seasonal behavior.

In [50]:
# ============================================================
# CHART 4 - Quarterly Seasonality Patterns
# Shows: How transactions vary across quarters within each
#        year and whether the pattern repeats year over year
# Insight target: Identify which quarter consistently
#                 drives the most transactions
# ============================================================

# Create a readable quarter label
df_quarterly['quarter_label'] = 'Q' + df_quarterly['quarter'].astype(str)
df_quarterly['transactions_billion'] = df_quarterly['total_transactions'] / 1e9

fig = go.Figure()

# One line per year
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', 
          '#FFA15A', '#19D3F3', '#FF6692']

for i, year in enumerate(sorted(df_quarterly['year'].unique())):
    df_year = df_quarterly[df_quarterly['year'] == year]
    fig.add_trace(
        go.Scatter(
            x=df_year['quarter_label'],
            y=df_year['transactions_billion'],
            name=str(year),
            mode='lines+markers+text',
            line=dict(color=colors[i], width=2),
            marker=dict(size=8),
            text=df_year['transactions_billion'].round(1),
            textposition='top center'
        )
    )

fig.update_layout(
    title={
        'text': 'Quarterly Transaction Trends by Year (2018—2024)',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    xaxis_title='Quarter',
    yaxis_title='Total Transactions (Billions)',
    legend_title='Year'
)

fig.show()

---
# 3. Case Study 2 — Device Dominance and User Engagement

## Business Context
Understanding which device brands PhonePe users prefer helps the company
make decisions on app optimization, device manufacturer partnerships,
and targeted marketing. We also analyze which states have the highest
and lowest user engagement to identify retention opportunities.

We analyze:
- Device brand market share nationally
- App engagement ratio across states
- Device brand growth trends over time
- Relationship between registered users and app opens

In [51]:
# ============================================================
# CASE STUDY 2 - DATA LOADING
# Purpose: Load all data needed for Case Study 2 charts
# ============================================================

with engine.connect() as conn:

    # Query 1 data -- device brand distribution
    df_devices = pd.read_sql(text("""
        SELECT
            device_brand,
            SUM(device_count) AS total_users,
            ROUND(
                100.0 * SUM(device_count) / SUM(SUM(device_count)) OVER(),
                2
            ) AS percentage_share,
            ROUND((AVG(device_percentage) * 100)::NUMERIC, 2) AS avg_usage_percentage
        FROM aggregated_user
        WHERE state != 'india'
        GROUP BY device_brand
        ORDER BY total_users DESC
    """), conn)

    # Query 2 data -- state engagement ratio
    df_engagement = pd.read_sql(text("""
        SELECT
            state,
            SUM(registered_users) AS total_registered,
            SUM(app_opens) AS total_app_opens,
            ROUND(
                (SUM(app_opens)::NUMERIC / NULLIF(SUM(registered_users), 0)),
                2
            ) AS avg_opens_per_user
        FROM aggregated_user
        WHERE state != 'india'
        GROUP BY state
        ORDER BY avg_opens_per_user DESC
    """), conn)

    # Query 3 data -- device brand year over year
    df_device_yoy = pd.read_sql(text("""
        SELECT
            year,
            device_brand,
            SUM(device_count) AS total_users
        FROM aggregated_user
        WHERE state != 'india'
        AND device_brand IN ('Xiaomi', 'Samsung', 'Vivo', 'Oppo', 'Realme')
        GROUP BY year, device_brand
        ORDER BY year ASC, total_users DESC
    """), conn)

    # Extra query -- registered users vs app opens by state
    df_reg_vs_opens = pd.read_sql(text("""
        SELECT
            state,
            MAX(registered_users) AS peak_registered_users,
            SUM(app_opens) AS total_app_opens
        FROM aggregated_user
        WHERE state != 'india'
        GROUP BY state
        ORDER BY peak_registered_users DESC
        LIMIT 15
    """), conn)

# Clean state names
df_engagement['state_clean'] = df_engagement['state'].str.replace('-', ' ').str.title()
df_reg_vs_opens['state_clean'] = df_reg_vs_opens['state'].str.replace('-', ' ').str.title()

print(" Case Study 2 data loaded successfully")
print(f"  Device data          : {len(df_devices)} rows")
print(f"  Engagement data      : {len(df_engagement)} rows")
print(f"  Device YoY data      : {len(df_device_yoy)} rows")
print(f"  Reg vs Opens data    : {len(df_reg_vs_opens)} rows")

 Case Study 2 data loaded successfully
  Device data          : 20 rows
  Engagement data      : 36 rows
  Device YoY data      : 25 rows
  Reg vs Opens data    : 15 rows


### Chart 5 — Device Brand Market Share
**Chart type:** Treemap  
**Why this chart:** Treemap is ideal for showing hierarchical proportional 
data where we have many categories of very different sizes. A pie chart 
with 20 brands becomes unreadable. A treemap naturally handles small 
categories by making th

In [52]:
# ============================================================
# CHART 5 - Device Brand Market Share Treemap
# Shows: Proportional share of each device brand among
#        PhonePe users nationally
# Insight target: Xiaomi dominance and Apple's surprisingly
#                 low share despite premium positioning
# ============================================================

fig = px.treemap(
    df_devices,
    path=['device_brand'],
    values='total_users',
    color='percentage_share',
    color_continuous_scale='Blues',
    title='PhonePe User Distribution by Device Brand',
    custom_data=['percentage_share', 'total_users']
)

fig.update_traces(
    texttemplate='<b>%{label}</b><br>%{customdata[0]}%',
    textfont_size=14
)

fig.update_layout(
    title={
        'text': 'PhonePe User Distribution by Device Brand',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    coloraxis_colorbar=dict(title='Share %')
)

fig.show()


### Chart 6 — User Engagement by State
**Chart type:** Horizontal bar chart with color gradient  
**Why this chart:** We have 36 states to compare on a single metric — 
app opens per user. Horizontal bars handle long state names cleanly. 
A color gradient from low to high engagement immediately shows the 
geographic pattern without needing to read every bar carefully.

In [53]:
# ============================================================
# CHART 6 - User Engagement by State
# Shows: Average app opens per registered user across all
#        states -- reveals which states are most engaged
# Insight target: Southern states dominate engagement while
#                 NCR states (Delhi, Gujarat) underperform
# ============================================================

# Sort by engagement for better visual
df_eng_sorted = df_engagement.sort_values('avg_opens_per_user', ascending=True)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=df_eng_sorted['state_clean'],
        x=df_eng_sorted['avg_opens_per_user'],
        orientation='h',
        marker=dict(
            color=df_eng_sorted['avg_opens_per_user'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Opens per User')
        ),
        text=df_eng_sorted['avg_opens_per_user'].round(1),
        textposition='outside'
    )
)

# Add a vertical line at the median engagement
median_eng = df_engagement['avg_opens_per_user'].median()
fig.add_vline(
    x=median_eng,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: {median_eng:.1f}',
    annotation_position='top'
)

fig.update_layout(
    title={
        'text': 'Average App Opens per Registered User by State',
        'font': {'size': 18},
        'x': 0.5
    },
    height=700,
    template='plotly_dark',
    xaxis_title='Average App Opens per User',
    yaxis_title='State'
)

fig.show()

### Chart 7 — Device Brand Growth Trends Over Time
**Chart type:** Line chart with multiple lines (one per brand)  
**Why this chart:** We need to track 5 brands across 5 years simultaneously. 
A grouped bar chart would work but becomes cluttered with 25 bars. 
A multi-line chart shows trajectory and crossing points clearly — 
specifically where Vivo overtook Samsung in 2021.

**Note:** Device data is only available until 2022 due to a change in 
PhonePe's data reporting methodology. This is a data availability 
limitation, not a code error.

In [54]:
# ============================================================
# CHART 7 - Device Brand Growth Trends Over Time
# Shows: How top 5 device brand user counts changed from
#        2018 to 2022
# Insight target: Vivo's rapid rise nearly overtaking
#                 Samsung by 2021
# Note: Data stops at 2022 due to reporting change
# ============================================================

df_device_yoy['users_million'] = df_device_yoy['total_users'] / 1e6

colors = {
    'Xiaomi': '#FF6B00',
    'Samsung': '#1428A0',
    'Vivo': '#415FFF',
    'Oppo': '#1D8348',
    'Realme': '#FFD700'
}

fig = go.Figure()

for brand in ['Xiaomi', 'Samsung', 'Vivo', 'Oppo', 'Realme']:
    df_brand = df_device_yoy[df_device_yoy['device_brand'] == brand]
    fig.add_trace(
        go.Scatter(
            x=df_brand['year'],
            y=df_brand['users_million'],
            name=brand,
            mode='lines+markers+text',
            line=dict(color=colors[brand], width=3),
            marker=dict(size=10),
            text=df_brand['users_million'].round(1),
            textposition='top center'
        )
    )

# Highlight the 2021-2022 drop with annotation
fig.add_annotation(
    x=2022,
    y=88,
    text='⚠️ Sharp drop in 2022<br>due to reporting change',
    showarrow=True,
    arrowhead=2,
    arrowcolor='red',
    font=dict(color='red', size=11),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.update_layout(
    title={
        'text': 'Top 5 Device Brand User Growth 2018—2022',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Total Users (Millions)',
    legend_title='Device Brand'
)

fig.show()

### Chart 8 — Registered Users vs App Opens by State
**Chart type:** Scatter plot  
**Why this chart:** We want to show the relationship between two continuous 
variables — registered users and app opens — across 15 states simultaneously. 
A scatter plot reveals clusters, outliers, and correlation patterns that 
bar charts cannot show. States far from the trend line are the most 
interesting business cases.

In [55]:
# ============================================================
# CHART 8 - Registered Users vs App Opens Scatter Plot
# Shows: Relationship between user registration numbers
#        and actual app engagement across top 15 states
# Insight target: States that punch above their weight
#                 in engagement vs their registration size
# ============================================================

df_reg_vs_opens['registered_million'] = df_reg_vs_opens['peak_registered_users'] / 1e6
df_reg_vs_opens['opens_billion'] = df_reg_vs_opens['total_app_opens'] / 1e9

fig = px.scatter(
    df_reg_vs_opens,
    x='registered_million',
    y='opens_billion',
    text='state_clean',
    size='registered_million',
    color='opens_billion',
    color_continuous_scale='Viridis',
    title='Registered Users vs App Opens — Top 15 States'
)

fig.update_traces(
    textposition='top center',
    marker=dict(sizemode='area', sizeref=0.1)
)

# Add trend line manually
import numpy as np
x = df_reg_vs_opens['registered_million']
y = df_reg_vs_opens['opens_billion']
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)

fig.add_trace(
    go.Scatter(
        x=x_line,
        y=p(x_line),
        mode='lines',
        name='Trend Line',
        line=dict(color='red', dash='dash', width=2)
    )
)

fig.update_layout(
    height=550,
    template='plotly_dark',
    xaxis_title='Peak Registered Users (Millions)',
    yaxis_title='Total App Opens (Billions)',
    title={
        'text': 'Registered Users vs App Opens — Top 15 States',
        'font': {'size': 18},
        'x': 0.5
    }
)

fig.show()

---
# 4. Case Study 3 — Insurance Penetration and Growth Potential

## Business Context
PhonePe entered the insurance market and needs to understand its growth 
trajectory, identify which states are leading adoption, and find untapped 
markets where high transaction activity exists but insurance usage is low.

**Important data note:** Insurance data starts from 2020 Q2. All 2020 
figures are understated due to missing Q1 data. Growth rates involving 
2020 should be interpreted with this caveat.

We analyze:
- Year over year insurance growth trajectory
- Top states by insurance adoption
- Insurance penetration rate vs transaction volume
- Quarterly insurance trends

In [56]:
# ============================================================
# CASE STUDY 3 - DATA LOADING
# Purpose: Load all data needed for Case Study 3 charts
# ============================================================

with engine.connect() as conn:

    # Query 1 data -- insurance YoY growth
    df_ins_yoy = pd.read_sql(text("""
        SELECT
            year,
            SUM(transaction_count) AS total_insurance_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_insurance_amount,
            ROUND(
                100.0 * (SUM(transaction_count) - LAG(SUM(transaction_count)) OVER (ORDER BY year))
                / NULLIF(LAG(SUM(transaction_count)) OVER (ORDER BY year), 0),
                2
            ) AS yoy_growth_percent
        FROM aggregated_insurance
        WHERE state != 'india'
        GROUP BY year
        ORDER BY year
    """), conn)

    # Query 2 data -- top states by insurance
    df_ins_states = pd.read_sql(text("""
        SELECT
            state,
            SUM(transaction_count) AS total_insurance_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount,
            ROUND(
                100.0 * SUM(transaction_count) / SUM(SUM(transaction_count)) OVER(),
                2
            ) AS percentage_share
        FROM aggregated_insurance
        WHERE state != 'india'
        GROUP BY state
        ORDER BY total_insurance_transactions DESC
        LIMIT 10
    """), conn)

    # Query 3 data -- insurance penetration rate
    df_ins_penetration = pd.read_sql(text("""
        SELECT
            t.state,
            SUM(t.transaction_count) AS regular_transactions,
            SUM(i.transaction_count) AS insurance_transactions,
            ROUND(
                100.0 * SUM(i.transaction_count) /
                NULLIF(SUM(t.transaction_count), 0),
                4
            ) AS insurance_penetration_rate
        FROM aggregated_transaction t
        LEFT JOIN aggregated_insurance i
            ON t.state = i.state
            AND t.year = i.year
            AND t.quarter = i.quarter
        WHERE t.state != 'india'
        AND t.year >= 2020
        GROUP BY t.state
        ORDER BY insurance_penetration_rate DESC
    """), conn)

    # Extra query -- quarterly insurance trends
    df_ins_quarterly = pd.read_sql(text("""
        SELECT
            year,
            quarter,
            SUM(transaction_count) AS total_insurance_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount
        FROM aggregated_insurance
        WHERE state != 'india'
        GROUP BY year, quarter
        ORDER BY year, quarter
    """), conn)

# Clean state names
df_ins_states['state_clean'] = df_ins_states['state'].str.replace('-', ' ').str.title()
df_ins_penetration['state_clean'] = df_ins_penetration['state'].str.replace('-', ' ').str.title()

print(" Case Study 3 data loaded successfully")
print(f"  Insurance YoY data        : {len(df_ins_yoy)} rows")
print(f"  Insurance states data     : {len(df_ins_states)} rows")
print(f"  Penetration rate data     : {len(df_ins_penetration)} rows")
print(f"  Quarterly insurance data  : {len(df_ins_quarterly)} rows")

 Case Study 3 data loaded successfully
  Insurance YoY data        : 5 rows
  Insurance states data     : 10 rows
  Penetration rate data     : 36 rows
  Quarterly insurance data  : 19 rows


### Chart 9 — Insurance Growth Trajectory
**Chart type:** Dual axis line + bar chart  
**Why this chart:** Same reasoning as Chart 3 — we need to show absolute 
volume growth (bar) and growth rate percentage (line) simultaneously. 
The two metrics have completely different scales so dual axis is essential. 
This directly mirrors the transaction growth chart allowing visual 
comparison between the two businesses.

In [57]:
# ============================================================
# CHART 9 - Insurance Growth Trajectory
# Shows: Absolute insurance transaction volume AND growth
#        rate percentage year over year
# Insight target: Insurance growing faster in value than
#                 volume -- users buying expensive policies
# Note: 2020 data starts from Q2 only -- growth rate
#       for 2021 is overstated due to incomplete 2020 base
# ============================================================

fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- Bars: Absolute insurance volume ---
fig.add_trace(
    go.Bar(
        x=df_ins_yoy['year'],
        y=df_ins_yoy['total_insurance_transactions'],
        name='Insurance Transactions',
        marker_color='#00CC96',
        text=df_ins_yoy['total_insurance_transactions'].apply(
            lambda x: f'{x/1e6:.2f}M'
        ),
        textposition='outside'
    ),
    secondary_y=False
)

# --- Line: YoY growth rate ---
df_ins_growth = df_ins_yoy.dropna(subset=['yoy_growth_percent'])

fig.add_trace(
    go.Scatter(
        x=df_ins_growth['year'],
        y=df_ins_growth['yoy_growth_percent'],
        name='YoY Growth Rate (%)',
        mode='lines+markers+text',
        line=dict(color='#EF553B', width=3),
        marker=dict(size=10),
        text=df_ins_growth['yoy_growth_percent'].astype(str) + '%',
        textposition='top center'
    ),
    secondary_y=True
)

# Add annotation for data quality caveat
fig.add_annotation(
    x=2021,
    y=100.86,
    text='⚠️ 2020 Q1 missing<br>growth overstated',
    showarrow=True,
    arrowhead=2,
    arrowcolor='yellow',
    font=dict(color='yellow', size=10),
    bgcolor='rgba(0,0,0,0.5)',
    yref='y2'
)

fig.update_layout(
    title={
        'text': 'PhonePe Insurance Growth 2020—2024',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

fig.update_xaxes(title_text='Year')
fig.update_yaxes(title_text='Total Insurance Transactions', secondary_y=False)
fig.update_yaxes(title_text='YoY Growth Rate (%)', secondary_y=True)

fig.show()

### Chart 10 — Top 10 States by Insurance Adoption
**Chart type:** Horizontal bar chart with dual metrics  
**Why this chart:** We need to compare both transaction count and amount 
across 10 states. Horizontal bars handle state names cleanly. Showing 
both metrics side by side reveals states where high volume doesn't 
always mean high value — pointing to different insurance product mixes.

In [58]:
# ============================================================
# CHART 10 - Top 10 States by Insurance Adoption
# Shows: Which states lead in insurance transactions
#        and insurance transaction value
# Insight target: Tamil Nadu jumps to 3rd in insurance
#                 despite not appearing in top 10 for
#                 regular transactions
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Top 10 States — Insurance Transaction Count',
        'Top 10 States — Insurance Amount (₹ Billion)'
    )
)

df_ins_states['amount_billion'] = df_ins_states['total_amount'] / 1e9

# Sort separately for each chart
df_ins_vol = df_ins_states.sort_values(
    'total_insurance_transactions', ascending=True
)
df_ins_val = df_ins_states.sort_values(
    'amount_billion', ascending=True
)

# --- Left: Transaction count ---
fig.add_trace(
    go.Bar(
        y=df_ins_vol['state_clean'],
        x=df_ins_vol['total_insurance_transactions'],
        orientation='h',
        marker_color='#00CC96',
        text=df_ins_vol['total_insurance_transactions'].apply(
            lambda x: f'{x/1e6:.2f}M'
        ),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Transaction amount ---
fig.add_trace(
    go.Bar(
        y=df_ins_val['state_clean'],
        x=df_ins_val['amount_billion'],
        orientation='h',
        marker_color='#AB63FA',
        text=df_ins_val['amount_billion'].round(2),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'Top 10 States — Insurance Transaction Analysis',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark'
)

fig.update_xaxes(title_text='Insurance Transactions', row=1, col=1)
fig.update_xaxes(title_text='Amount (₹ Billion)', row=1, col=2)

fig.show()

### Chart 11 — Insurance Penetration Rate Across All States
**Chart type:** Horizontal bar chart with color gradient  
**Why this chart:** We need to show all 36 states ranked by penetration 
rate. This is the most important insurance chart because it reveals 
untapped markets — states with high transaction activity but very low 
insurance adoption. A color gradient immediately highlights the extremes.

In [59]:
# ============================================================
# CHART 11 - Insurance Penetration Rate Across All States
# Shows: Insurance transactions as a percentage of regular
#        transactions for every state
# Insight target: Telangana has lowest penetration despite
#                 being a top transaction state -- biggest
#                 untapped insurance market in India
# ============================================================

# Sort ascending -- lowest penetration at top
df_pen_sorted = df_ins_penetration.sort_values(
    'insurance_penetration_rate', ascending=True
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=df_pen_sorted['state_clean'],
        x=df_pen_sorted['insurance_penetration_rate'],
        orientation='h',
        marker=dict(
            color=df_pen_sorted['insurance_penetration_rate'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title='Penetration %')
        ),
        text=df_pen_sorted['insurance_penetration_rate'].apply(
            lambda x: f'{x:.4f}%'
        ),
        textposition='outside'
    )
)

# Add median line
median_pen = df_ins_penetration['insurance_penetration_rate'].median()
fig.add_vline(
    x=median_pen,
    line_dash='dash',
    line_color='white',
    annotation_text=f'Median: {median_pen:.4f}%',
    annotation_position='top'
)

fig.update_layout(
    title={
        'text': 'Insurance Penetration Rate by State (% of Regular Transactions)',
        'font': {'size': 18},
        'x': 0.5
    },
    height=800,
    template='plotly_dark',
    xaxis_title='Insurance Penetration Rate (%)',
    yaxis_title='State'
)

fig.show()

### Chart 12 — Quarterly Insurance Trends
**Chart type:** Heatmap  
**Why this chart:** A heatmap is ideal for showing a two dimensional 
pattern — year on one axis, quarter on the other, with color intensity 
showing transaction volume. It immediately reveals both growth over time 
and seasonal patterns within each year in a single compact visual.

In [60]:
# ============================================================
# CHART 12 - Quarterly Insurance Trends Heatmap
# Shows: Insurance transaction volume across every year
#        and quarter combination as a color intensity map
# Insight target: Which quarters drive insurance purchases
#                 and how has the pattern changed over years
# ============================================================

# Pivot data for heatmap format
df_ins_pivot = df_ins_quarterly.pivot(
    index='year',
    columns='quarter',
    values='total_insurance_transactions'
)

# Rename columns to readable quarter labels
df_ins_pivot.columns = [f'Q{q}' for q in df_ins_pivot.columns]

fig = go.Figure(
    data=go.Heatmap(
        z=df_ins_pivot.values,
        x=df_ins_pivot.columns.tolist(),
        y=df_ins_pivot.index.tolist(),
        colorscale='Viridis',
        text=[[
            f'{v/1e6:.2f}M' if not pd.isna(v) else 'No Data'
            for v in row
        ] for row in df_ins_pivot.values],
        texttemplate='%{text}',
        textfont=dict(size=13),
        colorbar=dict(title='Transactions')
    )
)

# Add annotation for missing Q1 2020
fig.add_annotation(
    x='Q1',
    y=2020,
    text='⚠️ Missing',
    showarrow=False,
    font=dict(color='red', size=11)
)

fig.update_layout(
    title={
        'text': 'Insurance Transactions Heatmap — Year vs Quarter',
        'font': {'size': 18},
        'x': 0.5
    },
    height=450,
    template='plotly_dark',
    xaxis_title='Quarter',
    yaxis_title='Year'
)

fig.show()

---
# 5. Case Study 7 — Transaction Analysis Across States and Districts

## Business Context
Going below state level to district and pin code granularity reveals 
PhonePe's true geographic concentration. A state may rank highly overall 
but have all its transactions concentrated in one or two urban districts. 
Understanding this helps PhonePe target merchant acquisition and 
marketing at the exact right locations.

**Important data note:** Hyderabad district shows an anomalous spike in 
2022 followed by a sharp drop in 2023 — likely due to district boundary 
reclassification. Rangareddy district shows the inverse pattern confirming 
this hypothesis.

We analyze:
- Top districts by transaction volume and value nationally
- Top pin codes by transaction volume
- District level growth trajectories over time
- Transaction concentration within top states

In [61]:
# ============================================================
# CASE STUDY 7 - DATA LOADING
# Purpose: Load all data needed for Case Study 7 charts
# ============================================================

with engine.connect() as conn:

    # Query 1 data -- top districts nationally
    df_districts = pd.read_sql(text("""
        SELECT
            state,
            district,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount
        FROM map_transaction
        GROUP BY state, district
        ORDER BY total_transactions DESC
        LIMIT 15
    """), conn)

    # Query 2 data -- top pin codes
    df_pincodes = pd.read_sql(text("""
        SELECT
            state,
            entity_name AS pincode,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount
        FROM top_transaction
        WHERE entity_type = 'pincode'
        GROUP BY state, entity_name
        ORDER BY total_transactions DESC
        LIMIT 10
    """), conn)

    # Query 3 data -- district growth over time
    df_district_growth = pd.read_sql(text("""
        SELECT
            year,
            district,
            state,
            SUM(transaction_count) AS total_transactions,
            ROUND(SUM(transaction_amount)::NUMERIC, 2) AS total_amount
        FROM map_transaction
        WHERE district IN (
            'bengaluru urban district',
            'pune district',
            'hyderabad district',
            'jaipur district',
            'rangareddy district'
        )
        GROUP BY year, district, state
        ORDER BY district, year
    """), conn)

    # Extra query -- transaction concentration within top states
    df_concentration = pd.read_sql(text("""
        SELECT
            state,
            district,
            SUM(transaction_count) AS total_transactions,
            ROUND(
                100.0 * SUM(transaction_count) /
                SUM(SUM(transaction_count)) OVER (PARTITION BY state),
                2
            ) AS state_share_percent
        FROM map_transaction
        WHERE state IN (
            'karnataka', 'maharashtra',
            'telangana', 'rajasthan', 'andhra-pradesh'
        )
        GROUP BY state, district
        ORDER BY state, total_transactions DESC
    """), conn)

# Clean labels
df_districts['state_clean'] = df_districts['state'].str.replace(
    '-', ' ').str.title()
df_districts['district_clean'] = df_districts['district'].str.replace(
    '-', ' ').str.title()
df_districts['label'] = df_districts['district_clean'] + '\n(' + \
    df_districts['state_clean'] + ')'

df_pincodes['state_clean'] = df_pincodes['state'].str.replace(
    '-', ' ').str.title()

df_district_growth['district_clean'] = df_district_growth[
    'district'].str.replace('-', ' ').str.title()

print(" Case Study 7 data loaded successfully")
print(f"  District data        : {len(df_districts)} rows")
print(f"  Pincode data         : {len(df_pincodes)} rows")
print(f"  District growth data : {len(df_district_growth)} rows")
print(f"  Concentration data   : {len(df_concentration)} rows")

 Case Study 7 data loaded successfully
  District data        : 15 rows
  Pincode data         : 10 rows
  District growth data : 35 rows
  Concentration data   : 189 rows


### Chart 13 — Top 15 Districts by Transaction Volume
**Chart type:** Horizontal bar chart with state color coding  
**Why this chart:** We have 15 districts from multiple states. Color 
coding by state immediately shows which states dominate at district 
level without needing to read every label carefully. Horizontal bars 
handle long district names cleanly.

In [62]:
# ============================================================
# CHART 13 - Top 15 Districts by Transaction Volume
# Shows: Which districts nationally lead in transaction
#        volume with state level color coding
# Insight target: Bengaluru Urban alone outperforms entire
#                 states -- extreme geographic concentration
# ============================================================

df_dist_sorted = df_districts.sort_values(
    'total_transactions', ascending=True
)
df_dist_sorted['transactions_billion'] = \
    df_dist_sorted['total_transactions'] / 1e9
df_dist_sorted['amount_trillion'] = \
    df_dist_sorted['total_amount'] / 1e12

# Assign colors by state
state_colors = {
    'Karnataka': '#636EFA',
    'Maharashtra': '#EF553B',
    'Telangana': '#00CC96',
    'Rajasthan': '#AB63FA',
    'Andhra Pradesh': '#FFA15A',
    'Odisha': '#19D3F3',
    'Haryana': '#FF6692',
    'Uttar Pradesh': '#B6E880'
}

df_dist_sorted['color'] = df_dist_sorted['state_clean'].map(
    lambda x: state_colors.get(x, '#FFFFFF')
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=df_dist_sorted['district_clean'],
        x=df_dist_sorted['transactions_billion'],
        orientation='h',
        marker_color=df_dist_sorted['color'],
        text=df_dist_sorted['transactions_billion'].round(1),
        textposition='outside',
        customdata=df_dist_sorted['state_clean'],
        hovertemplate='<b>%{y}</b><br>State: %{customdata}<br>' +
                      'Transactions: %{x:.1f}B<extra></extra>'
    )
)

# Add legend manually for states
for state, color in state_colors.items():
    if state in df_dist_sorted['state_clean'].values:
        fig.add_trace(
            go.Bar(
                y=[None],
                x=[None],
                name=state,
                marker_color=color,
                orientation='h'
            )
        )

fig.update_layout(
    title={
        'text': 'Top 15 Districts — Transaction Volume (Billions)',
        'font': {'size': 18},
        'x': 0.5
    },
    height=600,
    template='plotly_dark',
    xaxis_title='Total Transactions (Billions)',
    yaxis_title='District',
    legend_title='State',
    barmode='overlay'
)

fig.show()

### Chart 14 — Top 10 Pin Codes by Transaction Volume
**Chart type:** Bar chart with state color coding  
**Why this chart:** Pin codes are the most granular geographic unit 
in this dataset. A bar chart with state color coding shows both the 
ranking and the geographic distribution of top pin codes simultaneously. 
This is the most actionable chart for PhonePe's merchant acquisition 
team — these are exact locations to deploy resources.

In [63]:
# ============================================================
# CHART 14 - Top 10 Pin Codes by Transaction Volume
# Shows: Which specific pin codes generate the most
#        transactions nationally
# Insight target: Telangana pin codes dominate top spots
#                 NCR pin codes appear despite low state
#                 level engagement
# ============================================================

df_pincodes['transactions_billion'] = \
    df_pincodes['total_transactions'] / 1e9
df_pincodes['amount_trillion'] = \
    df_pincodes['total_amount'] / 1e12
df_pincodes['label'] = df_pincodes['pincode'].astype(str) + \
    ' (' + df_pincodes['state_clean'] + ')'

# Sort descending for bar chart
df_pin_sorted = df_pincodes.sort_values(
    'transactions_billion', ascending=False
)

# Assign colors by state
df_pin_sorted['color'] = df_pin_sorted['state_clean'].map(
    lambda x: state_colors.get(x, '#FFFFFF')
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Top 10 Pin Codes — Transaction Count (Billions)',
        'Top 10 Pin Codes — Transaction Amount (₹ Trillion)'
    )
)

# --- Left: Transaction count ---
fig.add_trace(
    go.Bar(
        x=df_pin_sorted['label'],
        y=df_pin_sorted['transactions_billion'],
        marker_color=df_pin_sorted['color'],
        text=df_pin_sorted['transactions_billion'].round(2),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Transaction amount ---
df_pin_val = df_pincodes.sort_values('amount_trillion', ascending=False)
df_pin_val['label'] = df_pin_val['pincode'].astype(str) + \
    ' (' + df_pin_val['state_clean'] + ')'
df_pin_val['color'] = df_pin_val['state_clean'].map(
    lambda x: state_colors.get(x, '#FFFFFF')
)

fig.add_trace(
    go.Bar(
        x=df_pin_val['label'],
        y=df_pin_val['amount_trillion'],
        marker_color=df_pin_val['color'],
        text=df_pin_val['amount_trillion'].round(2),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'Top 10 Pin Codes — Transaction Volume vs Value',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark'
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_yaxes(title_text='Transactions (Billions)', row=1, col=1)
fig.update_yaxes(title_text='Amount (₹ Trillion)', row=1, col=2)

fig.show()

### Chart 15 — District Growth Trajectories Over Time
**Chart type:** Line chart with multiple lines (one per district)  
**Why this chart:** We need to track 5 districts across 7 years 
simultaneously to identify growth trajectories and anomalies. 
A line chart shows continuity and crossing points clearly. 
This is the only chart type that can reveal the Hyderabad 
district boundary reclassification anomaly visually.

In [64]:
# ============================================================
# CHART 15 - District Growth Trajectories Over Time
# Shows: Year over year transaction growth for top 5
#        districts from 2018 to 2024
# Insight target: Bengaluru Urban's consistent growth vs
#                 Hyderabad's anomalous 2022 spike and drop
#                 confirming district boundary reclassification
# ============================================================

df_district_growth['transactions_billion'] = \
    df_district_growth['total_transactions'] / 1e9

district_colors = {
    'Bengaluru Urban District': '#636EFA',
    'Pune District': '#EF553B',
    'Hyderabad District': '#00CC96',
    'Jaipur District': '#AB63FA',
    'Rangareddy District': '#FFA15A'
}

fig = go.Figure()

for district in df_district_growth['district_clean'].unique():
    df_d = df_district_growth[
        df_district_growth['district_clean'] == district
    ].sort_values('year')

    fig.add_trace(
        go.Scatter(
            x=df_d['year'],
            y=df_d['transactions_billion'],
            name=district,
            mode='lines+markers+text',
            line=dict(
                color=district_colors.get(district, '#FFFFFF'),
                width=3
            ),
            marker=dict(size=10),
            text=df_d['transactions_billion'].round(1),
            textposition='top center'
        )
    )

# Annotate Hyderabad anomaly
fig.add_annotation(
    x=2022,
    y=2.97,
    text='⚠️ Boundary<br>reclassification',
    showarrow=True,
    arrowhead=2,
    arrowcolor='yellow',
    font=dict(color='yellow', size=10),
    bgcolor='rgba(0,0,0,0.5)'
)

# Annotate Rangareddy surge
fig.add_annotation(
    x=2023,
    y=1.61,
    text='📈 Rangareddy<br>absorbs Hyderabad<br>districts',
    showarrow=True,
    arrowhead=2,
    arrowcolor='orange',
    font=dict(color='orange', size=10),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.update_layout(
    title={
        'text': 'Top 5 Districts — Transaction Growth 2018—2024',
        'font': {'size': 18},
        'x': 0.5
    },
    height=550,
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Total Transactions (Billions)',
    legend_title='District'
)

fig.show()

### Chart 16 — Transaction Concentration Within Top States
**Chart type:** Treemap with state and district hierarchy  
**Why this chart:** A hierarchical treemap is the only chart that can 
show two levels simultaneously — state level and district level — in 
a single visual. It immediately reveals which states have concentrated 
transactions in few districts vs spread across many districts. This 
tells us about urban vs distributed market structures.

In [65]:
# ============================================================
# CHART 16 - Transaction Concentration Within Top States
# Shows: How transactions are distributed across districts
#        within each of the top 5 states
# Insight target: Some states are dominated by one city
#                 while others have more distributed markets
# ============================================================

df_concentration['transactions_billion'] = \
    df_concentration['total_transactions'] / 1e9

# Clean labels
df_concentration['state_clean'] = df_concentration['state'].str.replace(
    '-', ' ').str.title()
df_concentration['district_clean'] = df_concentration['district'].str.replace(
    '-', ' ').str.title()

# Keep top 5 districts per state for readability
df_conc_top = df_concentration.groupby('state_clean').apply(
    lambda x: x.nlargest(5, 'total_transactions')
).reset_index(drop=True)

fig = px.treemap(
    df_conc_top,
    path=['state_clean', 'district_clean'],
    values='transactions_billion',
    color='state_share_percent',
    color_continuous_scale='RdYlGn',
    title='Transaction Concentration — Top 5 Districts per State',
    custom_data=['state_share_percent']
)

fig.update_traces(
    texttemplate='<b>%{label}</b><br>%{customdata[0]:.1f}%',
    textfont_size=13
)

fig.update_layout(
    title={
        'text': 'Transaction Concentration — Top 5 Districts per State',
        'font': {'size': 18},
        'x': 0.5
    },
    height=600,
    template='plotly_dark',
    coloraxis_colorbar=dict(title='State Share %')
)

fig.show()

---
# 6. Case Study 8 — User Registration Analysis

## Business Context
Understanding where PhonePe's registered users are coming from and 
whether those users are actually engaged helps the company distinguish 
between vanity metrics (registration count) and real business metrics 
(active engaged users). This analysis identifies both growth markets 
and retention risk markets.

**Important data note:** registered_users is a cumulative metric — 
it represents total users registered up to that point, not new users 
that quarter. We use MAX() to get the peak registered user count 
per state rather than SUM() which would inflate numbers incorrectly.

We analyze:
- Top states by registered users and engagement ratio
- Top districts by registered users
- Top pin codes by registered users
- Registration growth over time for top states

In [66]:
# ============================================================
# CASE STUDY 8 - DATA LOADING
# Purpose: Load all data needed for Case Study 8 charts
# ============================================================

with engine.connect() as conn:

    # Query 1 data -- top states by registered users
    df_reg_states = pd.read_sql(text("""
        SELECT
            state,
            MAX(registered_users) AS peak_registered_users,
            SUM(app_opens) AS total_app_opens,
            ROUND(
                (SUM(app_opens)::NUMERIC / NULLIF(MAX(registered_users), 0)),
                2
            ) AS opens_per_user
        FROM aggregated_user
        WHERE state != 'india'
        GROUP BY state
        ORDER BY peak_registered_users DESC
        LIMIT 10
    """), conn)

    # Query 2 data -- top districts by registered users
    df_reg_districts = pd.read_sql(text("""
        SELECT
            state,
            district,
            MAX(registered_users) AS peak_registered_users,
            SUM(app_opens) AS total_app_opens,
            ROUND(
                (SUM(app_opens)::NUMERIC / NULLIF(MAX(registered_users), 0)),
                2
            ) AS opens_per_user
        FROM map_user
        GROUP BY state, district
        ORDER BY peak_registered_users DESC
        LIMIT 10
    """), conn)

    # Query 3 data -- top pin codes by registered users
    df_reg_pincodes = pd.read_sql(text("""
        SELECT
            state,
            entity_name AS pincode,
            SUM(registered_users) AS total_registered_users
        FROM top_user
        WHERE entity_type = 'pincode'
        GROUP BY state, entity_name
        ORDER BY total_registered_users DESC
        LIMIT 10
    """), conn)

    # Extra query -- registration growth over time top 5 states
    df_reg_growth = pd.read_sql(text("""
        SELECT
            state,
            year,
            MAX(registered_users) AS peak_registered_users
        FROM aggregated_user
        WHERE state IN (
            'maharashtra', 'uttar-pradesh', 'karnataka',
            'andhra-pradesh', 'rajasthan'
        )
        GROUP BY state, year
        ORDER BY state, year
    """), conn)

# Clean labels
df_reg_states['state_clean'] = df_reg_states['state'].str.replace(
    '-', ' ').str.title()
df_reg_districts['state_clean'] = df_reg_districts['state'].str.replace(
    '-', ' ').str.title()
df_reg_districts['district_clean'] = df_reg_districts['district'].str.replace(
    '-', ' ').str.title()
df_reg_pincodes['state_clean'] = df_reg_pincodes['state'].str.replace(
    '-', ' ').str.title()
df_reg_pincodes['label'] = df_reg_pincodes['pincode'].astype(str) + \
    ' (' + df_reg_pincodes['state_clean'] + ')'
df_reg_growth['state_clean'] = df_reg_growth['state'].str.replace(
    '-', ' ').str.title()

print(" Case Study 8 data loaded successfully")
print(f"  Registered states data   : {len(df_reg_states)} rows")
print(f"  Registered districts data: {len(df_reg_districts)} rows")
print(f"  Registered pincodes data : {len(df_reg_pincodes)} rows")
print(f"  Registration growth data : {len(df_reg_growth)} rows")

 Case Study 8 data loaded successfully
  Registered states data   : 10 rows
  Registered districts data: 10 rows
  Registered pincodes data : 10 rows
  Registration growth data : 25 rows


### Chart 17 — Top 10 States by Registered Users and Engagement
**Chart type:** Grouped bar chart with dual metrics  
**Why this chart:** We need to compare two completely different metrics 
— registered users (absolute count) and opens per user (ratio) — across 
10 states simultaneously. A grouped bar chart puts both metrics side by 
side for each state making the registration vs engagement gap immediately 
visible for every state at once.

In [67]:
# ============================================================
# CHART 17 - Top 10 States by Registered Users and Engagement
# Shows: Registration numbers vs engagement ratio side by
#        side for top 10 states by user base
# Insight target: Maharashtra leads registrations but
#                 Andhra Pradesh and Rajasthan lead engagement
#                 revealing retention risk in large states
# ============================================================

df_reg_states_sorted = df_reg_states.sort_values(
    'peak_registered_users', ascending=False
)
df_reg_states_sorted['users_million'] = \
    df_reg_states_sorted['peak_registered_users'] / 1e6

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Peak Registered Users (Millions)',
        'App Opens per Registered User'
    )
)

# --- Left: Registered users ---
fig.add_trace(
    go.Bar(
        x=df_reg_states_sorted['state_clean'],
        y=df_reg_states_sorted['users_million'],
        marker_color='#636EFA',
        text=df_reg_states_sorted['users_million'].round(1),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Engagement ratio ---
# Color code by engagement -- red for low, green for high
df_reg_states_sorted['eng_color'] = df_reg_states_sorted[
    'opens_per_user'].apply(
    lambda x: '#00CC96' if x >= 3000 else
              '#FFA15A' if x >= 2000 else '#EF553B'
)

fig.add_trace(
    go.Bar(
        x=df_reg_states_sorted['state_clean'],
        y=df_reg_states_sorted['opens_per_user'],
        marker_color=df_reg_states_sorted['eng_color'],
        text=df_reg_states_sorted['opens_per_user'].apply(
            lambda x: f'{x:,.0f}'
        ),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

# Add median line on engagement chart
median_opens = df_reg_states['opens_per_user'].median()
fig.add_hline(
    y=median_opens,
    line_dash='dash',
    line_color='white',
    annotation_text=f'Median: {median_opens:,.0f}',
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'Top 10 States — Registered Users vs Engagement Ratio',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark'
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_yaxes(title_text='Users (Millions)', row=1, col=1)
fig.update_yaxes(title_text='Opens per User', row=1, col=2)

fig.show()

### Chart 18 — Top 10 Districts by Registered Users
**Chart type:** Horizontal bar chart with engagement overlay  
**Why this chart:** We need to show both registration count and 
engagement ratio for 10 districts simultaneously. Horizontal bars 
handle long district names. Color coding by engagement ratio on the 
same chart reveals the registration vs engagement gap at district 
level — the same story as Chart 17 but at finer granularity.

In [68]:
# ============================================================
# CHART 18 - Top 10 Districts by Registered Users
# Shows: Registration numbers and engagement ratio for
#        top 10 districts by user base
# Insight target: Ahmedabad spelling inconsistency creates
#                 two separate entries -- data quality issue
#                 Chennai and Ahmedabad have shockingly low
#                 engagement despite large user bases
# ============================================================

df_reg_dist_sorted = df_reg_districts.sort_values(
    'peak_registered_users', ascending=True
)
df_reg_dist_sorted['users_million'] = \
    df_reg_dist_sorted['peak_registered_users'] / 1e6

# Flag Ahmedabad data quality issue
df_reg_dist_sorted['district_label'] = df_reg_dist_sorted.apply(
    lambda row: row['district_clean'] + ' ⚠️'
    if 'ahmadabad' in row['district'].lower() or
       'ahmedabad' in row['district'].lower()
    else row['district_clean'],
    axis=1
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Peak Registered Users (Millions)',
        'App Opens per Registered User'
    )
)

# --- Left: Registered users ---
fig.add_trace(
    go.Bar(
        y=df_reg_dist_sorted['district_label'],
        x=df_reg_dist_sorted['users_million'],
        orientation='h',
        marker_color='#636EFA',
        text=df_reg_dist_sorted['users_million'].round(1),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# --- Right: Engagement ratio ---
df_reg_dist_sorted['eng_color'] = df_reg_dist_sorted[
    'opens_per_user'].apply(
    lambda x: '#00CC96' if x >= 400 else
              '#FFA15A' if x >= 250 else '#EF553B'
)

fig.add_trace(
    go.Bar(
        y=df_reg_dist_sorted['district_label'],
        x=df_reg_dist_sorted['opens_per_user'],
        orientation='h',
        marker_color=df_reg_dist_sorted['eng_color'],
        text=df_reg_dist_sorted['opens_per_user'].apply(
            lambda x: f'{x:,.0f}'
        ),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(
    title={
        'text': 'Top 10 Districts — Registered Users vs Engagement',
        'font': {'size': 18},
        'x': 0.5
    },
    height=550,
    template='plotly_dark'
)

fig.update_xaxes(title_text='Users (Millions)', row=1, col=1)
fig.update_xaxes(title_text='Opens per User', row=1, col=2)

fig.show()

### Chart 19 — Top 10 Pin Codes by Registered Users
**Chart type:** Horizontal bar chart with state color coding  
**Why this chart:** Pin codes are the most granular and actionable 
geographic unit. Color coding by state immediately shows geographic 
clustering — whether top registration pin codes are spread across 
India or concentrated in specific regions. This is the most 
actionable chart for PhonePe's user acquisition field teams.

In [69]:
# ============================================================
# CHART 19 - Top 10 Pin Codes by Registered Users
# Shows: Which specific pin codes have the highest
#        registered user counts nationally
# Insight target: NCR dominates registrations -- Noida,
#                 Delhi, Gurugram -- despite low engagement
#                 at state level confirming registration
#                 without retention problem in NCR
# ============================================================

df_reg_pincodes['users_million'] = \
    df_reg_pincodes['total_registered_users'] / 1e6

df_pin_reg_sorted = df_reg_pincodes.sort_values(
    'users_million', ascending=True
)

# Color by state
pin_state_colors = {
    'Uttar Pradesh': '#636EFA',
    'Delhi': '#EF553B',
    'Karnataka': '#00CC96',
    'Maharashtra': '#AB63FA',
    'Telangana': '#FFA15A',
    'Haryana': '#19D3F3'
}

df_pin_reg_sorted['color'] = df_pin_reg_sorted['state_clean'].map(
    lambda x: pin_state_colors.get(x, '#FFFFFF')
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=df_pin_reg_sorted['label'],
        x=df_pin_reg_sorted['users_million'],
        orientation='h',
        marker_color=df_pin_reg_sorted['color'],
        text=df_pin_reg_sorted['users_million'].round(2),
        textposition='outside',
        customdata=df_pin_reg_sorted['state_clean'],
        hovertemplate='<b>Pin: %{y}</b><br>State: %{customdata}' +
                      '<br>Users: %{x:.2f}M<extra></extra>'
    )
)

# Add legend for states
for state, color in pin_state_colors.items():
    if state in df_pin_reg_sorted['state_clean'].values:
        fig.add_trace(
            go.Bar(
                y=[None],
                x=[None],
                name=state,
                marker_color=color,
                orientation='h'
            )
        )

fig.update_layout(
    title={
        'text': 'Top 10 Pin Codes — Registered Users',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    xaxis_title='Registered Users (Millions)',
    yaxis_title='Pin Code (State)',
    legend_title='State',
    barmode='overlay'
)

fig.show()

**Observation:** Noida pin code 201301 leads nationally with 16.8M 
registered users. Delhi has 3 pin codes in top 10 — 110059, 110092, 
110086. Haryana has 2 pin codes — 122001 (Gurugram) and 121004 (Faridabad). 
NCR region dominates registration numbers controlling 7 out of top 10 
pin codes nationally.

**Business Insight:** NCR's dominance in registrations but absence from 
top transaction and engagement charts confirms a fundamental market 
problem — PhonePe successfully acquired users in NCR but failed to 
make them habitual. NCR users registered early when PhonePe launched 
aggressive acquisition campaigns but defaulted back to competitor apps 
for daily usage. This is a classic acquisition vs retention failure.

**Business Impact:** The NCR opportunity is enormous precisely because 
the hard part — registration — is already done. PhonePe does not need 
to spend acquisition budget in NCR. It needs to spend retention budget. 
Targeted re-engagement campaigns in these specific 7 NCR pin codes 
with strong daily use case incentives — metro card top-ups, grocery 
cashback, utility bill reminders — could unlock millions of dormant 
users at a fraction of new user acquisition cost.

### Chart 20 — Registration Growth Over Time for Top 5 States
**Chart type:** Line chart with multiple lines (one per state)  
**Why this chart:** We need to track how registered user bases have 
grown across 5 states over time. A multi-line chart shows both the 
absolute gap between states and the growth trajectory of each. 
This reveals which states are still growing rapidly vs which have 
plateaued — critical for resource allocation decisions.


In [70]:
# ============================================================
# CHART 20 - Registration Growth Over Time Top 5 States
# Shows: How registered user counts grew from 2018 to 2022
#        for the 5 largest state user bases
# Note: aggregated_user data only available until 2022
# Insight target: Maharashtra's lead is shrinking --
#                 Karnataka and Andhra Pradesh closing gap
# ============================================================

df_reg_growth['users_million'] = \
    df_reg_growth['peak_registered_users'] / 1e6

state_line_colors = {
    'Maharashtra': '#EF553B',
    'Uttar Pradesh': '#636EFA',
    'Karnataka': '#00CC96',
    'Andhra Pradesh': '#AB63FA',
    'Rajasthan': '#FFA15A'
}

fig = go.Figure()

for state in df_reg_growth['state_clean'].unique():
    df_s = df_reg_growth[
        df_reg_growth['state_clean'] == state
    ].sort_values('year')

    fig.add_trace(
        go.Scatter(
            x=df_s['year'],
            y=df_s['users_million'],
            name=state,
            mode='lines+markers+text',
            line=dict(
                color=state_line_colors.get(state, '#FFFFFF'),
                width=3
            ),
            marker=dict(size=10),
            text=df_s['users_million'].round(1),
            textposition='top center'
        )
    )

# Add annotation for data cutoff
fig.add_annotation(
    x=2022,
    y=2,
    text='⚠️ Data ends 2022<br>reporting change',
    showarrow=False,
    font=dict(color='yellow', size=11),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.update_layout(
    title={
        'text': 'Registered User Growth 2018—2022 — Top 5 States',
        'font': {'size': 18},
        'x': 0.5
    },
    height=500,
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Registered Users (Millions)',
    legend_title='State'
)

fig.show()

**Observation:** Maharashtra consistently leads registered users growing 
from a small base in 2018 to 48M by 2022. Karnataka shows the steepest 
growth trajectory among top 5 states — growing faster than Uttar Pradesh 
despite having a smaller population. Andhra Pradesh and Rajasthan show 
parallel growth curves suggesting similar adoption patterns.

**Business Insight:** Karnataka's fast registration growth combined with 
its high engagement ratio from Chart 17 makes it the most valuable 
state in PhonePe's portfolio — high registrations AND high engagement. 
Uttar Pradesh's slower registration growth relative to its massive 
population confirms that rural UP remains largely untapped — there is 
significant room for growth in India's most populous state.

**Business Impact:** Karnataka should be PhonePe's showcase market — 
case studies, testimonials, and merchant success stories from Karnataka 
should be used in marketing campaigns targeting similar southern states. 
Uttar Pradesh needs a dedicated rural expansion strategy — partnership 
with local kirana stores, agricultural payment solutions, and regional 
language support to penetrate beyond the major cities.

---
# 7. Cross Case Study Insights

## Business Context
Individual case studies reveal patterns within their domain. But the 
most powerful insights come from connecting findings across domains — 
transactions, users, and insurance together. These two charts synthesize 
everything we have found into a single unified business picture.

In [71]:
# ============================================================
# CROSS CASE STUDY - DATA LOADING
# Purpose: Load combined data that connects multiple tables
#          to reveal cross domain insights
# ============================================================

with engine.connect() as conn:

    # Combined state level summary across all domains
    df_combined = pd.read_sql(text("""
        SELECT
            t.state,
            SUM(t.transaction_count) AS total_transactions,
            ROUND(SUM(t.transaction_amount)::NUMERIC, 2) AS total_amount,
            MAX(u.registered_users) AS peak_registered_users,
            ROUND(
                (SUM(u.app_opens)::NUMERIC /
                NULLIF(MAX(u.registered_users), 0)),
                2
            ) AS opens_per_user,
            COALESCE(SUM(i.transaction_count), 0) AS insurance_transactions,
            ROUND(
                100.0 * COALESCE(SUM(i.transaction_count), 0) /
                NULLIF(SUM(t.transaction_count), 0),
                4
            ) AS insurance_penetration
        FROM aggregated_transaction t
        LEFT JOIN aggregated_user u
            ON t.state = u.state
            AND t.year = u.year
            AND t.quarter = u.quarter
        LEFT JOIN aggregated_insurance i
            ON t.state = i.state
            AND t.year = i.year
            AND t.quarter = i.quarter
        WHERE t.state != 'india'
        AND t.year >= 2020
        GROUP BY t.state
        ORDER BY total_transactions DESC
        LIMIT 15
    """), conn)

# Clean labels
df_combined['state_clean'] = df_combined['state'].str.replace(
    '-', ' ').str.title()
df_combined['transactions_billion'] = \
    df_combined['total_transactions'] / 1e9
df_combined['users_million'] = \
    df_combined['peak_registered_users'] / 1e6

print(" Cross case study data loaded successfully")
print(f"  Combined data: {len(df_combined)} rows")

 Cross case study data loaded successfully
  Combined data: 15 rows


### Chart 21 — Transaction Volume vs User Engagement Bubble Chart
**Chart type:** Bubble chart  
**Why this chart:** A bubble chart can show four dimensions simultaneously — 
x axis (transactions), y axis (engagement), bubble size (registered users), 
and color (insurance penetration). No other chart type can show this many 
variables at once. This is the single most information-dense chart in 
the entire notebook.

In [72]:
# ============================================================
# CHART 21 - Transaction Volume vs Engagement Bubble Chart
# Shows: Four metrics simultaneously per state --
#        transaction volume, engagement ratio, user base
#        size, and insurance penetration
# Insight target: Identify which states are strong across
#                 ALL metrics vs which have specific gaps
# ============================================================

fig = px.scatter(
    df_combined,
    x='transactions_billion',
    y='opens_per_user',
    size='users_million',
    color='insurance_penetration',
    text='state_clean',
    color_continuous_scale='RdYlGn',
    title='State Performance Matrix — Transactions vs Engagement',
    labels={
        'transactions_billion': 'Total Transactions (Billions)',
        'opens_per_user': 'App Opens per User',
        'users_million': 'Registered Users (M)',
        'insurance_penetration': 'Insurance Penetration %'
    },
    custom_data=[
        'state_clean',
        'users_million',
        'insurance_penetration'
    ]
)

fig.update_traces(
    textposition='top center',
    marker=dict(sizemode='area', sizeref=0.05),
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Transactions: %{x:.1f}B<br>'
        'Opens/User: %{y:,.0f}<br>'
        'Users: %{customdata[1]:.1f}M<br>'
        'Insurance: %{customdata[2]:.4f}%'
        '<extra></extra>'
    )
)

# Add quadrant lines at median values
median_trans = df_combined['transactions_billion'].median()
median_eng = df_combined['opens_per_user'].median()

fig.add_vline(
    x=median_trans,
    line_dash='dash',
    line_color='white',
    opacity=0.5,
    annotation_text='Median Transactions',
    annotation_position='top'
)

fig.add_hline(
    y=median_eng,
    line_dash='dash',
    line_color='white',
    opacity=0.5,
    annotation_text='Median Engagement',
    annotation_position='right'
)

# Label quadrants
fig.add_annotation(
    x=df_combined['transactions_billion'].max() * 0.85,
    y=df_combined['opens_per_user'].max() * 0.95,
    text='⭐ Star Markets',
    showarrow=False,
    font=dict(color='#00CC96', size=12),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.add_annotation(
    x=df_combined['transactions_billion'].max() * 0.85,
    y=median_eng * 0.3,
    text='⚠️ Volume Without Engagement',
    showarrow=False,
    font=dict(color='#FFA15A', size=12),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.add_annotation(
    x=median_trans * 0.2,
    y=df_combined['opens_per_user'].max() * 0.95,
    text='💎 Hidden Gems',
    showarrow=False,
    font=dict(color='#636EFA', size=12),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.add_annotation(
    x=median_trans * 0.2,
    y=median_eng * 0.3,
    text='🔴 Needs Attention',
    showarrow=False,
    font=dict(color='#EF553B', size=12),
    bgcolor='rgba(0,0,0,0.5)'
)

fig.update_layout(
    height=600,
    template='plotly_dark',
    title={
        'text': 'State Performance Matrix — 4 Dimensions Simultaneously',
        'font': {'size': 18},
        'x': 0.5
    }
)

fig.show()

**Observation:** States fall into 4 distinct quadrants. Karnataka and 
Telangana are Star Markets — high transactions AND high engagement. 
Maharashtra and Uttar Pradesh are Volume Without Engagement markets — 
massive transaction bases but below median engagement. Andhra Pradesh 
and Rajasthan are Hidden Gems — high engagement despite moderate 
transaction volumes. Several smaller states fall in Needs Attention.

**Business Insight:** This matrix is the most actionable summary of 
PhonePe's entire market landscape. Each quadrant needs a completely 
different strategy — Star Markets need protection, Volume Without 
Engagement needs retention campaigns, Hidden Gems need transaction 
growth investment, Needs Attention needs fundamental market development.

**Business Impact:** PhonePe's leadership team can use this matrix 
as a strategic planning tool every quarter — tracking which states 
move between quadrants over time reveals whether business strategies 
are working. A state moving from Volume Without Engagement to Star 
Market confirms a successful retention campaign. A state moving from 
Hidden Gems to Needs Attention signals a competitive threat.

### Chart 22 — State Rankings Across All Three Domains
**Chart type:** Heatmap of rankings  
**Why this chart:** We have rankings for each state across three domains — 
transactions, users, and insurance. A heatmap of rankings shows at a 
glance which states are consistently strong across all domains vs which 
excel in one but underperform in others. Color intensity immediately 
highlights the best and worst performing states per domain.

In [73]:
# ============================================================
# CHART 22 - State Rankings Heatmap Across All Domains
# Shows: How each state ranks in transactions, registered
#        users, and insurance transactions simultaneously
# Insight target: No state dominates all three domains --
#                 every state has at least one weakness
#                 revealing targeted opportunity areas
# ============================================================

# Create rankings for each domain
df_combined['rank_transactions'] = df_combined[
    'total_transactions'].rank(ascending=False).astype(int)
df_combined['rank_users'] = df_combined[
    'peak_registered_users'].rank(ascending=False).astype(int)
df_combined['rank_insurance'] = df_combined[
    'insurance_transactions'].rank(ascending=False).astype(int)
df_combined['rank_engagement'] = df_combined[
    'opens_per_user'].rank(ascending=False).astype(int)

# Build ranking matrix
df_ranks = df_combined[[
    'state_clean',
    'rank_transactions',
    'rank_users',
    'rank_insurance',
    'rank_engagement'
]].set_index('state_clean')

df_ranks.columns = [
    'Transaction\nVolume',
    'Registered\nUsers',
    'Insurance\nAdoption',
    'User\nEngagement'
]

# Sort by average rank
df_ranks['avg_rank'] = df_ranks.mean(axis=1)
df_ranks = df_ranks.sort_values('avg_rank')
df_ranks = df_ranks.drop('avg_rank', axis=1)

fig = go.Figure(
    data=go.Heatmap(
        z=df_ranks.values,
        x=df_ranks.columns.tolist(),
        y=df_ranks.index.tolist(),
        colorscale='RdYlGn_r',
        text=df_ranks.values.astype(int),
        texttemplate='#%{text}',
        textfont=dict(size=14),
        colorbar=dict(
            title='Rank',
            tickvals=[1, 5, 10, 15],
            ticktext=['1st', '5th', '10th', '15th']
        )
    )
)

fig.update_layout(
    title={
        'text': 'State Performance Rankings Across All Domains',
        'font': {'size': 18},
        'x': 0.5
    },
    height=600,
    template='plotly_dark',
    xaxis_title='Domain',
    yaxis_title='State',
    xaxis=dict(side='top')
)

fig.show()

**Observation:** No state ranks 1st across all four domains simultaneously. 
Karnataka consistently ranks in top 3 across transactions, insurance, and 
engagement but ranks lower in registered users. Maharashtra leads registered 
users but ranks poorly in engagement. Telangana ranks highly in transactions 
and engagement but poorly in insurance — confirming our Case Study 3 finding.

**Business Insight:** The absence of a single dominant state across all 
metrics is actually healthy for PhonePe — it means no single state 
failure can collapse the entire business. However it also means no 
state has fully realized its potential across all product lines. 
Every state in this matrix has at least one domain where it underperforms 
relative to its overall size — representing a specific product opportunity.

**Business Impact:** This ranking matrix gives PhonePe's product teams 
a precise to-do list per state. Telangana — push insurance. Maharashtra — 
improve engagement. Karnataka — grow user registrations. Each state gets 
a targeted product intervention rather than generic national campaigns 
that dilute budget across all markets equally.

---
# 8. Business Recommendations and Conclusion

## Summary of Key Findings

After analyzing 116,749 rows of data across 9 tables covering transactions,
users, and insurance from 2018 to 2024, here are the most important 
findings from this analysis:

### Transaction Dynamics
- PhonePe processed 99 billion transactions worth ₹129 trillion in 2024
- Merchant payments dominate volume (55%) but P2P dominates value (₹266T)
- Growth rate is declining but absolute additions are accelerating
- Q4 consistently drives the highest transaction volumes due to festive season

### User Engagement
- Xiaomi, Samsung, and Vivo control 62% of PhonePe's user base
- Rajasthan and Andhra Pradesh lead engagement despite not being the 
  largest states
- NCR dominates registrations but has critically low engagement
- Chennai and Ahmedabad have the worst registration to engagement ratios

### Insurance
- Insurance grew 27x in value from 2020 to 2024 — outpacing volume growth
- Karnataka and Maharashtra lead adoption but Tamil Nadu punches above weight
- Telangana has the lowest insurance penetration despite being a top 
  transaction state
- All states have below 0.05% insurance penetration — massive untapped market

### Geographic Concentration
- Bengaluru Urban district alone outperforms entire states
- Top 5 districts account for a disproportionate share of national volume
- Rangareddy is the fastest emerging district nationally

---

## 5 Specific Business Recommendations

### Recommendation 1 — Launch Targeted Insurance Campaigns in Telangana
**Evidence:** Telangana ranks 3rd nationally in transaction volume but has 
the lowest insurance penetration rate of 0.0174%. Users are active and 
trust the platform but are not buying insurance.  
**Action:** Deploy in-app insurance nudges during high value transactions 
in Telangana. Offer first policy free or heavily discounted. Set a target 
of doubling Telangana's insurance penetration within 2 quarters.  
**Expected Impact:** Even moving Telangana from 0.0174% to 0.035% 
penetration would add approximately 4.5 million new insurance transactions 
annually given its transaction base.

### Recommendation 2 — Re-engagement Campaign for NCR Pin Codes
**Evidence:** 7 of the top 10 registered user pin codes are in NCR 
(Noida, Delhi, Gurugram, Faridabad) but Delhi's engagement ratio is 
only 16.5 opens per user — well below the national median of 22.  
**Action:** Run a 90 day re-engagement campaign targeting the 7 NCR 
pin codes with daily use case incentives — metro card top-ups, grocery 
cashback, utility bill reminders. No acquisition budget needed — 
users are already registered.  
**Expected Impact:** Converting 20% of dormant NCR users to active 
daily users would add tens of millions of daily transactions at near 
zero acquisition cost.

### Recommendation 3 — Pre-installation Partnership with Xiaomi and Vivo
**Evidence:** Xiaomi (25.13%) and Vivo (18.07%) together control 43% 
of PhonePe's entire user base. Realme is growing fastest at percentage 
level. All three are budget Android brands in the ₹10,000-₹20,000 segment.  
**Action:** Negotiate pre-installation agreements to make PhonePe a 
default app on new Xiaomi, Vivo, and Realme devices sold in India. 
Priority markets — UP, Bihar, Rajasthan where these brands dominate 
and PhonePe growth potential is highest.  
**Expected Impact:** Pre-installation typically drives 30-40% higher 
activation rates than organic installs. Given millions of Xiaomi and 
Vivo devices sold monthly in India this represents the highest ROI 
user acquisition channel available.

### Recommendation 4 — Merchant Acquisition Drive in Rangareddy District
**Evidence:** Rangareddy grew from 12M transactions in 2018 to 2.2B 
in 2024 — a 172x increase making it the fastest growing district 
nationally. It is currently in the exponential growth phase where 
early market dominance creates lasting competitive advantage.  
**Action:** Deploy a dedicated merchant acquisition team in Rangareddy 
for the next 2 quarters. Offer zero commission period for new merchants, 
QR code installation drives, and merchant training programs.  
**Expected Impact:** Winning merchant dominance in Rangareddy now 
while it is in exponential growth will compound over years as the 
district continues urbanizing. Missing this window means fighting 
an entrenched competitor later at much higher cost.

### Recommendation 5 — Q4 Insurance Bundle Campaign
**Evidence:** Insurance transactions spike in Q4 along with regular 
transactions. Users have festive season bonuses and are in a spending 
mindset. Insurance value per transaction has grown 27x suggesting 
users are willing to buy premium policies.  
**Action:** Launch a "Diwali Protection Bundle" in Q4 offering 
health, life, and vehicle insurance as a discounted package exclusively 
through PhonePe. Promote through in-app banners starting Q3 to build 
awareness before the Q4 purchase window.  
**Expected Impact:** A dedicated Q4 insurance campaign aligned with 
peak spending season could accelerate insurance penetration growth 
rate from current 27% annually to 40%+ in the campaign year.

---

## Conclusion

This analysis reveals that PhonePe is a mature, high scale payments 
platform that has successfully built transaction volume but faces 
three distinct strategic challenges going forward.

First, converting registered users into daily active users in large 
markets like Maharashtra, Delhi, Gujarat, and Chennai where registrations 
are high but engagement is low. Second, growing its insurance business 
from near-zero penetration in high transaction states — particularly 
Telangana, Madhya Pradesh, and Odisha — where the user base exists 
but insurance awareness does not. Third, defending its dominant position 
in Star Markets like Karnataka and Telangana against competitor encroachment 
while simultaneously developing emerging districts like Rangareddy.

The data consistently shows that PhonePe's strongest markets are not 
always its largest ones. Rajasthan and Andhra Pradesh punch above their 
weight in engagement. Tamil Nadu punches above its weight in insurance. 
Rangareddy punches above its weight in growth rate. These hidden 
strength markets deserve as much strategic attention as the obvious 
large markets of Maharashtra and Karnataka.

PhonePe's next phase of growth will not come from acquiring new users — 
it will come from maximizing the value of the 400+ million users it 
already has.